In [16]:
import pathlib
from model_functions import test
import torch
from medmnist import ChestMNIST
from torchvision.transforms import v2
from model import ResNet18

tf = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32,scale=True)
])

test_data = ChestMNIST(split='test',transform=tf,download=True,root="./data")

classes = len(test_data.info["label"])
channels = test_data.info["n_channels"]
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ResNet18(channels,classes).to(device)


In [17]:
local_test_results = []

node = pathlib.Path(f"./results_centralised").resolve()
model.load_state_dict(torch.load(node/"best_model.pth"))
print(f"Centralised Training")
test_dl = torch.utils.data.DataLoader(test_data,16384)
_,test_acc = test(test_dl,model,loss_fn=torch.nn.BCEWithLogitsLoss(),device=device)
local_test_results.append((f"centralised",test_acc))


Centralised Training
Test Error: Accuracy: 94.7%, Average Loss: 0.181169


In [18]:
import pandas as pd

pd.DataFrame(local_test_results)

,0,1
0,centralised,0.947437
